# Uji Model - Anggota 1 (Ketua)

**Mazmur Bryan**: EncoderCNN, LSTMCell manual, DecoderLSTM, CaptioningModel.

Semua modul dibangun from scratch (tanpa model pretrained, tanpa `nn.LSTM`) dan diuji dengan tensor acak, karena dataset belum dibutuhkan untuk memastikan arsitektur benar.

Cara menjalankan lokal: buka Jupyter dari root repo, pilih kernel `.venv`, lalu **Run → Run All Cells**.

## Setup

In [1]:
# Setup repo lokal. Jalankan Jupyter dari root repo atau folder notebooks.
from pathlib import Path
import os
import subprocess
import sys

candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
REPO_ROOT = next((p for p in candidates if (p / 'src').is_dir() and (p / 'tests').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Root repo tidak ditemukan. Jalankan Jupyter dari root repo atau folder notebooks.')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

def muat_ulang():
    for nama in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
        del sys.modules[nama]
    print('Modul src dimuat ulang pada import berikutnya')

print('Repo:', REPO_ROOT)
print('Python:', sys.executable)

Repo: C:\Users\zaidu\OneDrive\Documents\Kuliah\Semester 5\Generative AI\ImageCaptioningBackstreet
Python: C:\Users\zaidu\OneDrive\Documents\Kuliah\Semester 5\Generative AI\ImageCaptioningBackstreet\.venv\Scripts\python.exe


## Cek GPU

In [2]:
import torch
print('PyTorch:', torch.__version__)
print('GPU tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cpu
GPU tersedia: False


## 1. EncoderCNN

Vanilla CNN 5 blok (Conv 3×3 → BatchNorm → ReLU → MaxPool) + Global Average Pooling.
Gambar `(B, 3, 128, 128)` → fitur `(B, 256)`.

Yang dicek: shape tiap blok, keluaran `(4, 256)`, dan gradien mengalir sampai lapisan pertama.

In [3]:
subprocess.run([sys.executable, '-m', 'tests.test_encoder'], check=True)

CompletedProcess(args=['C:\\Users\\zaidu\\OneDrive\\Documents\\Kuliah\\Semester 5\\Generative AI\\ImageCaptioningBackstreet\\.venv\\Scripts\\python.exe', '-m', 'tests.test_encoder'], returncode=0)

## 2. LSTMCell manual

Sel LSTM ditulis sendiri dari `nn.Linear`: 4 gate (input, forget, candidate, output), bias forget gate = 1.

Yang dicek: shape, rumus gate dihitung ulang secara terpisah, bias forget, rentang `h`, dan gradien setelah 30 langkah.

In [4]:
subprocess.run([sys.executable, '-m', 'tests.test_lstm_cell'], check=True)

CompletedProcess(args=['C:\\Users\\zaidu\\OneDrive\\Documents\\Kuliah\\Semester 5\\Generative AI\\ImageCaptioningBackstreet\\.venv\\Scripts\\python.exe', '-m', 'tests.test_lstm_cell'], returncode=0)

## 3. DecoderLSTM

Embedding → LSTMCell → Dropout → Linear. Fitur gambar menjadi memori awal (`h0`, `c0`). Training memakai teacher forcing, pembuatan caption memakai greedy search.

Yang dicek: shape `(4, 29, vocab)`, tidak mengintip kata masa depan, fitur gambar dipakai, embedding `<pad>` = 0, gradien, dan `generate`.

In [5]:
subprocess.run([sys.executable, '-m', 'tests.test_decoder'], check=True)

CompletedProcess(args=['C:\\Users\\zaidu\\OneDrive\\Documents\\Kuliah\\Semester 5\\Generative AI\\ImageCaptioningBackstreet\\.venv\\Scripts\\python.exe', '-m', 'tests.test_decoder'], returncode=0)

## 4. CaptioningModel + uji hafalan

Encoder + decoder digabung dalam satu model. **Uji hafalan**: model dilatih 150 langkah pada 4 pasangan gambar-caption acak. Jika semua bagian tersambung benar, loss turun drastis dan model bisa menulis ulang caption yang dihafal.

In [6]:
subprocess.run([sys.executable, '-m', 'tests.test_model'], check=True)

CompletedProcess(args=['C:\\Users\\zaidu\\OneDrive\\Documents\\Kuliah\\Semester 5\\Generative AI\\ImageCaptioningBackstreet\\.venv\\Scripts\\python.exe', '-m', 'tests.test_model'], returncode=0)

## Ringkasan

Jika keempat uji menampilkan `LULUS`, arsitektur model siap dipakai untuk training dengan data Flickr8k (Anggota 2 & 3).